# 02 · Ingesta a Bronze

| | |
|---|---|
| **Fase CRISP-DM** | 2 — Comprensión de los datos (ingesta) |
| **Objetivo** | Cargar las dos hojas del Excel de origen **sin transformar valores**, validar su esquema (US001–US003) y guardarlas en la capa Bronze. |
| **Entradas** | `data/raw/VoltajesDiariosDesde2020.xlsx` (hojas `EL A 2020_2026`, `EL B 2020_2026`) |
| **Salidas** | `data/bronze/voltajes_EL_A.parquet`, `data/bronze/voltajes_EL_B.parquet`, `data/bronze/_metadata.json` |

**Reglas de la capa Bronze**
- Los valores quedan tal como vienen en el Excel: sin limpieza, sin reordenar y sin eliminar duplicados. Solo se descartan las filas completamente vacías del final de la hoja.
- Los nombres de columna se convierten a texto. Los encabezados numéricos de las celdas (`1.0`, `2.0`, …) quedan como `"1"`, `"2"`, …
- Se agregan las columnas de linaje `electrolizador` (A/B), `_source_file` e `_ingested_at`.
- Validaciones de esquema: existe `Date`, existen `[Volts]` y `[kA]` y hay columnas de celdas numéricas (se **cuentan**, no se asume 181). Si falla, se lanza un error descriptivo.

In [ ]:
# Configuración: localiza la raíz del proyecto (local o Colab) y agrega src/ al path
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/Poc-TP1")
except ImportError:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "voltcast").is_dir())
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
from datetime import datetime, timezone

import pandas as pd

from voltcast import config, io

config.ensure_dirs()
src = config.raw_path()
if not src.exists():
    raise FileNotFoundError(f"No se encontró el archivo de origen: {src}. Cópielo en {config.RAW_DIR}.")
src_hash = io.file_sha256(src)
print(f"Origen : {src.name} ({src.stat().st_size / 1e6:.1f} MB)")
print(f"SHA-256: {src_hash}")

## 1. Lectura y validación de esquema

In [ ]:
ingested_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
bronze, schemas = {}, {}

for el, sheet in config.SHEETS.items():
    df = io.read_excel_sheet(src, sheet)
    schema = io.detect_schema(df, sheet)  # SchemaError si falta algo (US001-E03 / US002-E02 / US003-E02)
    df["electrolizador"] = el
    df["_source_file"] = src.name
    df["_ingested_at"] = ingested_at
    bronze[el], schemas[el] = df, schema
    print(f"EL {el} · hoja '{sheet}': {len(df)} filas × {df.shape[1]} columnas")
    print(f"   fecha: '{schema.date_col}' | voltaje total: '{schema.volts_col}' | corriente: '{schema.ka_col}'")
    print(f"   celdas detectadas: {schema.n_cells} ('{schema.cell_cols[0]}' … '{schema.cell_cols[-1]}') | otras: {schema.other_cols}")

In [ ]:
# Resumen de la identificación de columnas (US002) y celdas (US003)
resumen = pd.DataFrame({
    el: {
        "filas": len(bronze[el]),
        "celdas": s.n_cells,
        "primera fecha": bronze[el][s.date_col].min(),
        "última fecha": bronze[el][s.date_col].max(),
        "otras columnas": ", ".join(s.other_cols),
    }
    for el, s in schemas.items()
}).T
resumen

In [ ]:
# Las celdas deben ser las mismas en ambos electrolizadores
if schemas["A"].cell_cols != schemas["B"].cell_cols:
    raise io.SchemaError("Las columnas de celdas de EL A y EL B no coinciden.")
bronze["A"].head(3)

## 2. Escritura en Bronze

In [ ]:
for el, df in bronze.items():
    path = io.write_layer(
        df, "bronze", io.bronze_name(el), source=src, source_hash=src_hash,
        extra={"sheet": config.SHEETS[el], "schema": io.schema_to_dict(schemas[el])},
    )
    print(f"✔ {path.relative_to(config.BASE_DIR)} ({path.stat().st_size / 1e6:.2f} MB)")

In [ ]:
# Verificación: releer Bronze y comparar con lo leído del Excel
for el, df in bronze.items():
    back = io.read_bronze(el)
    pd.testing.assert_frame_equal(back, df, check_dtype=False)
    print(f"EL {el}: Bronze idéntico al Excel ({back.shape[0]} filas × {back.shape[1]} columnas)")

meta = io.read_metadata("bronze")
{k: {kk: v[kk] for kk in ("rows", "columns", "source_file", "executed_at")} for k, v in meta.items()}